# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# ML-07 Section 1: define the transparent baseline rule

import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/Ravindrathalari06/flyrank-internship-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Rows:", len(df))
print("Baseline rule: visibility + freshness + position opportunity + depth gap")

print("\nReason codes:")
reason_code_list = [
    "stale_visible_page",
    "thin_visible_page",
    "page_one_opportunity",
    "low_ctr_visible_page",
    "general_refresh_review"
]

for code in reason_code_list:
    print("-", code)

# Safe fields used for ranking.
# No trend_direction or trend_pct is used in the score.
score_fields = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count",
    "ctr"
]

print("\nScore fields:", score_fields)
print("Label-derived fields used in score:", [])

Rows: 30000
Baseline rule: visibility + freshness + position opportunity + depth gap

Reason codes:
- stale_visible_page
- thin_visible_page
- page_one_opportunity
- low_ctr_visible_page
- general_refresh_review

Score fields: ['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count', 'ctr']
Label-derived fields used in score: []


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 2: build and save the ranked baseline queue

import os
import numpy as np
import pandas as pd

# Reload the raw dataset so this section can run independently.
url = "https://raw.githubusercontent.com/Ravindrathalari06/flyrank-internship-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

def percentile_rank(series):
    return series.rank(pct=True, method="average").fillna(0)

# Safe numeric copies for scoring.
impressions = df["impressions_90d"].fillna(df["impressions_90d"].median())
days_update = df["days_since_last_update"].fillna(
    df["days_since_last_update"].median()
)
avg_position = df["avg_position"].fillna(df["avg_position"].median())
word_count = df["word_count"].fillna(df["word_count"].median())
ctr = df["ctr"].fillna(df["ctr"].median())

# 1. Visibility: higher impressions = higher priority.
visibility_score = percentile_rank(np.log1p(impressions))

# 2. Freshness risk: longer update gap = higher priority.
freshness_risk_score = percentile_rank(days_update)

# 3. Position opportunity:
# Pages with a useful search position and reasonable visibility
# receive more priority.
position_component = (
    1 - percentile_rank(avg_position.clip(lower=1, upper=50))
)

position_opportunity_score = (
    position_component
    * visibility_score
    * (avg_position > 0).astype(int)
)

# 4. Depth gap:
# Lower word count among visible pages gets a modest priority boost.
depth_gap_score = (
    1 - percentile_rank(word_count)
) * visibility_score

# Final transparent baseline score.
df["baseline_action_score"] = (
    0.40 * visibility_score
    + 0.30 * freshness_risk_score
    + 0.25 * position_opportunity_score
    + 0.05 * depth_gap_score
).clip(0, 1)

df["visibility_score"] = visibility_score
df["freshness_risk_score"] = freshness_risk_score
df["position_opportunity_score"] = position_opportunity_score
df["depth_gap_score"] = depth_gap_score


# ---------------------------------------------------------
# Reason codes — IMPORTANT:
# Do not use trend_direction or trend_pct here.
# ---------------------------------------------------------
def get_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")

    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["impressions_90d"] >= 250
    ):
        reasons.append("page_one_opportunity")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(get_reason_codes, axis=1)

# Suggested action.
def suggested_action(row):
    reasons = set(row["reason_codes"].split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if "stale_visible_page" in reasons:
        return "refresh"

    if "page_one_opportunity" in reasons:
        return "refresh_and_review_position"

    return "monitor"


df["suggested_action"] = df.apply(suggested_action, axis=1)

# Rank all pages.
df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Create output directory.
output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

output_columns = [
    "content_id",
    "baseline_rank",
    "baseline_action_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "days_since_last_update"
]

baseline_queue = (
    df[output_columns]
    .sort_values("baseline_rank")
    .reset_index(drop=True)
)

baseline_queue.to_csv(output_path, index=False)

print("Baseline queue created.")
print("Rows:", len(baseline_queue))
print("Output:", output_path)
print("Top score:", round(baseline_queue["baseline_action_score"].max(), 4))
print("Median score:", round(baseline_queue["baseline_action_score"].median(), 4))

print("\nTop 10:")
display(baseline_queue.head(10))

Baseline queue created.
Rows: 30000
Output: work/outputs/baseline_action_score.csv
Top score: 0.9207
Median score: 0.4145

Top 10:


,content_id,baseline_rank,baseline_action_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action,impressions_90d,avg_position,ctr,word_count,days_since_last_update
0,content_03d2673b2553,1,0.920667,0.997633,0.8432,0.945557,0.645286,page_one_opportunity,refresh_and_review_position,143314,1.9,0.83,2840.0,104
1,content_399f4eed93b9,2,0.916632,0.996633,0.8432,0.908930,0.755730,page_one_opportunity|low_ctr_visible_page,refresh_and_review_ctr,127658,3.4,0.43,2602.0,104
2,content_6ac3ab740bbf,3,0.916249,0.948617,0.9911,0.814419,0.717360,page_one_opportunity|low_ctr_visible_page,refresh_and_review_ctr,22462,4.6,0.14,2606.0,106
3,content_9532f197bbc8,4,0.914125,0.999633,0.8432,0.945320,0.499633,page_one_opportunity,refresh_and_review_position,309192,2.0,0.87,NaN,104
4,content_c1143eda3230,5,0.912323,0.995433,0.8432,0.907835,0.684609,page_one_opportunity,refresh_and_review_position,112578,3.4,1.03,2756.0,104
5,content_5184b85dc6dd,6,0.910881,0.990200,0.8432,0.898970,0.741973,page_one_opportunity,refresh_and_review_position,73699,3.5,0.80,2622.0,104
6,content_681d93f6924d,7,0.910052,0.978300,0.8432,0.888166,0.874617,page_one_opportunity|low_ctr_visible_page,refresh_and_review_ctr,42310,3.5,0.13,1495.0,104
7,content_4d1fe5b32dc2,8,0.908123,0.994167,0.8432,0.930606,0.496901,page_one_opportunity,refresh_and_review_position,97999,2.5,0.52,NaN,104
8,content_07f2e7a6f38a,9,0.907356,0.994467,0.8432,0.927025,0.497051,page_one_opportunity,refresh_and_review_position,101078,2.7,0.85,NaN,104
9,content_e1501cdeca69,10,0.907076,0.994033,0.8432,0.872877,0.765671,page_one_opportunity|low_ctr_visible_page,refresh_and_review_ctr,96467,4.2,0.48,2571.0,104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 3: review the top 20 ranked pages

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    score = row["baseline_action_score"]

    if score >= 0.75:
        return "Higher rule-based priority; multiple observed signals support review."
    elif score >= 0.50:
        return "Moderate rule-based priority; review supporting signals."
    else:
        return "Lower rule-based priority within the selected top-20; verify manually."


def what_could_make_it_wrong(row):
    reasons = row["reason_codes"]

    if "stale_visible_page" in reasons:
        return "The page may have a legitimate reason to remain unchanged despite the update gap."

    if "thin_visible_page" in reasons:
        return "Short content may be appropriate for the search intent."

    if "low_ctr_visible_page" in reasons:
        return "Low CTR may reflect query intent, SERP layout, or other factors not represented here."

    if "page_one_opportunity" in reasons:
        return "The position may not represent a stable opportunity or may reflect a different search context."

    return "The rule may not capture important context that is unavailable in this dataset."


top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_could_make_it_wrong"] = top20.apply(
    what_could_make_it_wrong,
    axis=1
)

review_columns = [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "suggested_action",
    "reason_codes",
    "confidence_note",
    "what_could_make_it_wrong"
]

top20_review = top20[review_columns]

print("Top-20 review:")
display(top20_review)

print("\nTop-20 count:", len(top20_review))

Top-20 review:


,baseline_rank,content_id,baseline_action_score,suggested_action,reason_codes,confidence_note,what_could_make_it_wrong
0,1,content_03d2673b2553,0.920667,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
1,2,content_399f4eed93b9,0.916632,refresh_and_review_ctr,page_one_opportunity|low_ctr_visible_page,Higher rule-based priority; multiple observed ...,"Low CTR may reflect query intent, SERP layout,..."
2,3,content_6ac3ab740bbf,0.916249,refresh_and_review_ctr,page_one_opportunity|low_ctr_visible_page,Higher rule-based priority; multiple observed ...,"Low CTR may reflect query intent, SERP layout,..."
3,4,content_9532f197bbc8,0.914125,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
4,5,content_c1143eda3230,0.912323,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
5,6,content_5184b85dc6dd,0.910881,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
6,7,content_681d93f6924d,0.910052,refresh_and_review_ctr,page_one_opportunity|low_ctr_visible_page,Higher rule-based priority; multiple observed ...,"Low CTR may reflect query intent, SERP layout,..."
7,8,content_4d1fe5b32dc2,0.908123,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
8,9,content_07f2e7a6f38a,0.907356,refresh_and_review_position,page_one_opportunity,Higher rule-based priority; multiple observed ...,The position may not represent a stable opport...
9,10,content_e1501cdeca69,0.907076,refresh_and_review_ctr,page_one_opportunity|low_ctr_visible_page,Higher rule-based priority; multiple observed ...,"Low CTR may reflect query intent, SERP layout,..."



Top-20 count: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 4: identify weak picks and check for leakage

print("=== WEAK PICK CHECK ===")

weak_picks = baseline_queue[
    baseline_queue["reason_codes"] == "general_refresh_review"
].head(10)

print("Example weak/general picks:")
display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_codes",
            "suggested_action"
        ]
    ]
)

print("\nNumber of general-review pages:",
      (baseline_queue["reason_codes"] == "general_refresh_review").sum())


# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

print("\n=== LEAKAGE CHECK ===")

label_derived_fields = [
    "trend_direction",
    "trend_pct"
]

product_or_decision_fields = [
    "provider_used",
    "model_used",
    "client_id"
]

future_like_fields = [
    "future",
    "next_30d",
    "next_90d",
    "future_impressions",
    "future_clicks",
    "future_sessions"
]

score_fields_used = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count",
    "ctr"
]

print("Score fields used:")
print(score_fields_used)

print("\nLabel-derived fields used in score:")
print([
    field for field in label_derived_fields
    if field in score_fields_used
])

print("\nProduct/decision fields used in score:")
print([
    field for field in product_or_decision_fields
    if field in score_fields_used
])

print("\nFuture-like fields used in score:")
print([
    field for field in future_like_fields
    if field in score_fields_used
])

assert not any(
    field in score_fields_used
    for field in label_derived_fields
)

assert not any(
    field in score_fields_used
    for field in product_or_decision_fields
)

assert not any(
    field in score_fields_used
    for field in future_like_fields
)

print("\nResult: No checked label-derived, product/decision, or future-like fields are used in the ranking score.")
print("The baseline is a transparent decision-support ranking, not a causal or guaranteed prediction.")

=== WEAK PICK CHECK ===
Example weak/general picks:


,baseline_rank,content_id,baseline_action_score,reason_codes,suggested_action
881,882,content_482aff19e9cc,0.796364,general_refresh_review,monitor
1108,1109,content_50426bec207f,0.779395,general_refresh_review,monitor
1273,1274,content_97bb507def8a,0.766541,general_refresh_review,monitor
1294,1295,content_fcc4d5bd69d4,0.764251,general_refresh_review,monitor
1296,1297,content_47b8b12d581e,0.764166,general_refresh_review,monitor
1303,1304,content_595d69dd2627,0.763687,general_refresh_review,monitor
1342,1343,content_a965a1fc5544,0.760617,general_refresh_review,monitor
1364,1365,content_db581a0e8733,0.759192,general_refresh_review,monitor
1366,1367,content_07eb63474d33,0.758924,general_refresh_review,monitor
1404,1405,content_6e3312cb2476,0.756756,general_refresh_review,monitor



Number of general-review pages: 17809

=== LEAKAGE CHECK ===
Score fields used:
['impressions_90d', 'days_since_last_update', 'avg_position', 'word_count', 'ctr']

Label-derived fields used in score:
[]

Product/decision fields used in score:
[]

Future-like fields used in score:
[]

Result: No checked label-derived, product/decision, or future-like fields are used in the ranking score.
The baseline is a transparent decision-support ranking, not a causal or guaranteed prediction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.